# NEXUS on Google Colab

This notebook starts the NEXUS control plane. It does not contain an API key. The key cell uses `getpass` and exports the value only for the running process.

Free Claude Code and OpenCode are not simulated here. The control plane starts the real entrypoints when their runtimes are installed. This notebook does not call KiosAPI unless you later press **Live probe** in the UI.

Expected tools:

- NEXUS control plane: Python 3.11+
- Free Claude Code: Python 3.14.7 exactly. The optional cell installs it with `uv` if you choose to.
- OpenCode: Node.js and `@opencode/cli@2.0.27`


In [ ]:
import os, shutil, subprocess, sys
print("python", sys.version)
for name in ("node", "npm", "git"):
    print(name, shutil.which(name))


## 1. Get the source

If you already opened this notebook from a checkout, skip the clone. The OpenCode submodule is required.


In [ ]:
import os, subprocess
from pathlib import Path

REPO = Path("/content/knik")
if not (REPO / "nexus" / "__init__.py").exists():
    subprocess.check_call(["git", "clone", "--recurse-submodules", "https://github.com/betaanoiar1-gif/knik.git", str(REPO)])
else:
    subprocess.check_call(["git", "-C", str(REPO), "submodule", "update", "--init", "--depth", "1", "third_party/opencode"])
os.chdir(REPO)
print("root", REPO)
print("fcc", (REPO / "third_party/free-claude-code/pyproject.toml").exists())
print("opencode", (REPO / "third_party/opencode/packages/cli/package.json").exists())
print("skills", len(list((REPO / "third_party/skills").rglob("SKILL.md"))))


## 2. Install the control plane

This does not download model weights and does not install Python 3.14.7.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])


## 3. Optional: OpenCode binary

This installs the published CLI that matches the pinned source commit. It is about 200 MB.


In [ ]:
import subprocess, shutil
install = input("Install @opencode/cli@2.0.27? [y/N] ").strip().lower() == "y"
if install:
    subprocess.check_call(["npm", "install", "-g", "@opencode/cli@2.0.27"])
    print("opencode", shutil.which("opencode"))
else:
    print("skipped. Set NEXUS_OPENCODE_BIN later if you already have the binary.")


## 4. Optional: Python 3.14.7 for Free Claude Code

Free Claude Code refuses other Python versions. This cell is optional and can take several minutes. It was not completed in the environment where NEXUS was assembled.


In [ ]:
import os, subprocess, sys
install = input("Install Python 3.14.7 with uv for Free Claude Code? [y/N] ").strip().lower() == "y"
if install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    subprocess.check_call(["uv", "python", "install", "3.14.7"])
    out = subprocess.check_output(["uv", "python", "find", "3.14.7"], text=True).strip()
    os.environ["NEXUS_FCC_PYTHON"] = out
    print("NEXUS_FCC_PYTHON", out)
else:
    print("skipped. Free Claude Code will report runtime_ready=false until NEXUS_FCC_PYTHON points at 3.14.7.")


## 5. Secrets

Leave the prompt empty if you only want to inspect the UI and the dry-run request. A live probe is not sent by this notebook.


In [ ]:
import getpass, os, secrets
from pathlib import Path

key = getpass.getpass("KiosAPI key (empty to skip): ")
if key:
    os.environ["NEXUS_KIOSAPI_API_KEY"] = key
    print("key exported for this process. length", len(key))
else:
    os.environ.pop("NEXUS_KIOSAPI_API_KEY", None)
    print("no key. Dry probe still works. Live probe will be refused.")

os.environ["NEXUS_AUTH_TOKEN"] = secrets.token_urlsafe(24)
os.environ["NEXUS_DATA_DIR"] = "/content/nexus-data"
os.environ["NEXUS_ALLOWED_ROOTS"] = "/content/workspaces"
os.environ["NEXUS_LOOPBACK_OPEN"] = "0"
Path("/content/workspaces/demo").mkdir(parents=True, exist_ok=True)
print("control token created. It will be printed once in the next cell.")


## 6. Start NEXUS and print the Colab URL

The server binds `0.0.0.0` so the Colab proxy can reach it. The token is required. The child engines still bind `127.0.0.1`.


In [ ]:
import os, subprocess, time
from pathlib import Path

port = 8787
log = Path("/content/nexus-server.log")
proc = subprocess.Popen(
    ["nexus", "serve", "--host", "0.0.0.0", "--port", str(port)],
    stdout=log.open("w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
Path("/content/nexus.pid").write_text(str(proc.pid))
time.sleep(2)
print("pid", proc.pid, "log", log)
print("token", os.environ["NEXUS_AUTH_TOKEN"])
try:
    from google.colab.output import eval_js
    base = eval_js(f"google.colab.kernel.proxyPort({port})")
    token = os.environ["NEXUS_AUTH_TOKEN"]
    print("open", f"{base}/?token={token}")
except Exception as exc:
    print("not running inside Colab, or proxy lookup failed:", type(exc).__name__)
    print(f"open http://127.0.0.1:{port}/?token={os.environ['NEXUS_AUTH_TOKEN']}")


## 7. Diagnostics

`doctor` does not print the API key. `runtime_ready: false` means the interpreter or binary is missing. It does not mean the button is fake.


In [ ]:
import json, os, urllib.request
req = urllib.request.Request(
    "http://127.0.0.1:8787/api/diagnostics",
    headers={"Authorization": "Bearer " + os.environ["NEXUS_AUTH_TOKEN"]},
)
with urllib.request.urlopen(req, timeout=10) as resp:
    body = json.load(resp)
print(json.dumps(body, indent=2)[:4000])


## 8. Stop

This stops the NEXUS process group. Engine children started by that process receive the signal through the same group.


In [ ]:
import os, signal
from pathlib import Path
pid = int(Path("/content/nexus.pid").read_text().strip())
os.killpg(pid, signal.SIGTERM)
print("stopped", pid)
